# Emergency Intent Classification - DistilBERT (Colab)

This notebook trains an 8-class emergency intent classifier using `distilbert-base-uncased`, with weighted loss and exports to PyTorch + ONNX.

In [ ]:
!pip -q install -U transformers datasets accelerate evaluate scikit-learn onnx onnxruntime

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from dataclasses import dataclass
from typing import Dict, List
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    DistilBertForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MODEL_NAME = "distilbert-base-uncased"
CLASS_NAMES = [
    "medical",
    "fire",
    "violent_crime",
    "accident",
    "gas_hazard",
    "mental_health",
    "non_emergency",
    "unknown",
]
label2id = {name: i for i, name in enumerate(CLASS_NAMES)}
id2label = {i: name for name, i in label2id.items()}

OUT_DIR = "/content/redline_intent_model"
PT_EXPORT_DIR = os.path.join(OUT_DIR, "pytorch")
ONNX_EXPORT_PATH = os.path.join(OUT_DIR, "intent_model.onnx")
os.makedirs(PT_EXPORT_DIR, exist_ok=True)

print("CUDA available:", torch.cuda.is_available())
print("Classes:", CLASS_NAMES)

In [ ]:
def generate_synthetic_samples() -> pd.DataFrame:
    templates: Dict[str, List[str]] = {
        "medical": [
            "my father is having severe chest pain and cannot breathe",
            "someone collapsed and is unconscious please send ambulance",
            "there is heavy bleeding after a fall",
            "possible overdose at home we need medical help now",
            "patient is having seizure and not responding"
        ],
        "fire": [
            "the kitchen is on fire smoke everywhere",
            "apartment building has visible flames on second floor",
            "garage fire spreading quickly near gas cans",
            "forest edge is burning and wind is strong",
            "electrical panel sparked and caught fire"
        ],
        "violent_crime": [
            "i heard gunshots outside and people are screaming",
            "a man with a knife attacked someone on the street",
            "armed robbery in progress at the store",
            "domestic assault happening now please send police",
            "there was a stabbing near the bus stop"
        ],
        "accident": [
            "two cars crashed at the intersection one flipped",
            "motorcycle accident with injured rider on highway",
            "truck collision blocking both lanes",
            "pedestrian hit by a vehicle near school",
            "multi vehicle crash with smoke from engine"
        ],
        "gas_hazard": [
            "strong gas smell in basement and people feel dizzy",
            "possible gas leak near restaurant kitchen",
            "chemical fumes coming from factory vent",
            "hissing sound from gas pipe outside home",
            "carbon monoxide alarm is going off repeatedly"
        ],
        "mental_health": [
            "my friend says he wants to hurt himself tonight",
            "caller is in crisis and talking about suicide",
            "person is having a severe panic episode and shaking",
            "my sister is threatening self harm",
            "someone is extremely distressed and needs immediate support"
        ],
        "non_emergency": [
            "there is loud music from neighbors please file complaint",
            "streetlight is broken near my house",
            "i want information about parking rules",
            "noise complaint from construction after hours",
            "lost wallet report not urgent"
        ],
        "unknown": [
            "hello are you there i need help",
            "something strange happened i do not know what it is",
            "please call me back as soon as possible",
            "i cannot explain but it feels wrong",
            "unclear emergency need assistance"
        ],
    }

    class_counts = {
        "medical": 1000,
        "fire": 800,
        "violent_crime": 650,
        "accident": 750,
        "gas_hazard": 450,
        "mental_health": 500,
        "non_emergency": 400,
        "unknown": 250,
    }

    prefixes = [
        "urgent", "please", "now", "immediately", "right away",
        "this is serious", "need help", "dispatcher", "911", "emergency"
    ]

    suffixes = [
        "location downtown",
        "near main street",
        "caller is panicking",
        "multiple people involved",
        "unknown injuries",
        "situation escalating",
        "line is unstable",
        "response requested",
        "cannot wait",
        "details still coming",
    ]

    rows = []
    for label, count in class_counts.items():
        base = templates[label]
        for _ in range(count):
            text = random.choice(base)
            if random.random() < 0.7:
                text = f"{random.choice(prefixes)} {text}"
            if random.random() < 0.7:
                text = f"{text} {random.choice(suffixes)}"
            if random.random() < 0.15:
                text = text.replace("please", "pls").replace("you", "u")
            rows.append({"text": text.strip(), "label_name": label})

    df = pd.DataFrame(rows).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    return df

possible_real_paths = [
    "/content/emergency_intent.csv",
    "/content/intent_8class_dataset.csv",
    "/content/final_8class_dataset_clean.csv",
    "./datasets/intent_8class_dataset.csv",
]

df = None
for p in possible_real_paths:
    if os.path.exists(p):
        tmp = pd.read_csv(p)
        cols = [c.lower() for c in tmp.columns]
        if "text" in cols:
            text_col = tmp.columns[cols.index("text")]
        elif "transcript" in cols:
            text_col = tmp.columns[cols.index("transcript")]
        else:
            continue

        if "label" in cols:
            label_col = tmp.columns[cols.index("label")]
        elif "intent" in cols:
            label_col = tmp.columns[cols.index("intent")]
        elif "class" in cols:
            label_col = tmp.columns[cols.index("class")]
        else:
            continue

        tmp = tmp[[text_col, label_col]].rename(columns={text_col: "text", label_col: "label_name"})
        tmp["label_name"] = tmp["label_name"].astype(str).str.strip().str.lower()
        tmp = tmp[tmp["label_name"].isin(CLASS_NAMES)].dropna().reset_index(drop=True)
        if len(tmp) > 0:
            df = tmp
            print(f"Loaded real dataset from: {p} | rows: {len(df)}")
            break

if df is None:
    df = generate_synthetic_samples()
    print(f"Generated synthetic dataset | rows: {len(df)}")

df["label"] = df["label_name"].map(label2id)
df = df.dropna(subset=["text", "label"]).reset_index(drop=True)
df["text"] = df["text"].astype(str)
df["label"] = df["label"].astype(int)
print(df.head(3))
print(df["label_name"].value_counts())

In [ ]:
train_df, temp_df = train_test_split(
    df, test_size=0.2, random_state=SEED, stratify=df["label"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=SEED, stratify=temp_df["label"]
)

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df[["text", "label"]], preserve_index=False),
    "validation": Dataset.from_pandas(val_df[["text", "label"]], preserve_index=False),
    "test": Dataset.from_pandas(test_df[["text", "label"]], preserve_index=False),
})

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

tokenized = dataset.map(tokenize_batch, batched=True)
tokenized = tokenized.remove_columns(["text"])
tokenized.set_format(type="torch")

print(tokenized)

In [ ]:
train_labels = np.array(train_df["label"].tolist())
weights_np = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(CLASS_NAMES)),
    y=train_labels
)
class_weights = torch.tensor(weights_np, dtype=torch.float)
print("Class weights:", class_weights)

model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(CLASS_NAMES),
    id2label=id2label,
    label2id=label2id
)

@dataclass
class WeightedTrainer(Trainer):
    class_weights: torch.Tensor = None

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        cw = self.class_weights.to(logits.device)
        loss_fct = nn.CrossEntropyLoss(weight=cw)
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

In [ ]:
training_args = TrainingArguments(
    output_dir=os.path.join(OUT_DIR, "checkpoints"),
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
)

train_result = trainer.train()
print(train_result)

val_metrics = trainer.evaluate(tokenized["validation"])
print("Validation metrics:", val_metrics)

test_metrics = trainer.evaluate(tokenized["test"])
print("Test metrics:", test_metrics)

In [ ]:
trainer.save_model(PT_EXPORT_DIR)
tokenizer.save_pretrained(PT_EXPORT_DIR)

with open(os.path.join(PT_EXPORT_DIR, "label_map.json"), "w") as f:
    json.dump({"id2label": id2label, "label2id": label2id}, f, indent=2)

print("Saved PyTorch model to:", PT_EXPORT_DIR)

In [ ]:
import onnx
import onnxruntime as ort

export_model = DistilBertForSequenceClassification.from_pretrained(PT_EXPORT_DIR)
export_model.eval()

dummy = tokenizer("sample emergency text", return_tensors="pt", truncation=True, max_length=128)
input_names = ["input_ids", "attention_mask"]
output_names = ["logits"]
dynamic_axes = {
    "input_ids": {0: "batch_size", 1: "sequence"},
    "attention_mask": {0: "batch_size", 1: "sequence"},
    "logits": {0: "batch_size"},
}

with torch.no_grad():
    torch.onnx.export(
        export_model,
        (dummy["input_ids"], dummy["attention_mask"]),
        ONNX_EXPORT_PATH,
        input_names=input_names,
        output_names=output_names,
        dynamic_axes=dynamic_axes,
        opset_version=17,
        do_constant_folding=True,
    )

onnx_model = onnx.load(ONNX_EXPORT_PATH)
onnx.checker.check_model(onnx_model)

session = ort.InferenceSession(ONNX_EXPORT_PATH, providers=["CPUExecutionProvider"])
ort_inputs = {
    "input_ids": dummy["input_ids"].cpu().numpy(),
    "attention_mask": dummy["attention_mask"].cpu().numpy(),
}
ort_out = session.run(None, ort_inputs)

print("ONNX export path:", ONNX_EXPORT_PATH)
print("ONNX logits shape:", np.array(ort_out[0]).shape)

In [ ]:
sample_texts = [
    "there is a gas leak in my apartment building",
    "my friend is unconscious and not breathing",
    "loud music complaint from neighbors",
]

infer_model = DistilBertForSequenceClassification.from_pretrained(PT_EXPORT_DIR)
infer_model.eval()

for t in sample_texts:
    enc = tokenizer(t, return_tensors="pt", truncation=True, max_length=128)
    with torch.no_grad():
        logits = infer_model(**enc).logits
        pred = torch.argmax(logits, dim=-1).item()
    print(t, "=>", id2label[pred])

print("Done. Artifacts:")
print("PyTorch:", PT_EXPORT_DIR)
print("ONNX:", ONNX_EXPORT_PATH)